<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Classical_ML_for_Code_Switching_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup, Imports, and Mock Data Creation

We create a simplified dataset structure directly using Python lists and convert it to a Pandas DataFrame.

In [2]:
# 0. SETUP AND IMPORTS
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ----------------------------------------------------------------------
# 1. MOCK DATA CREATION
# This simulates the structure of the tokens and tags from the real dataset.
# ----------------------------------------------------------------------

# Utterances (rows)
mock_data = [
    # Utterance 1: Mostly English, switch to Zh noun (intra-sentential)
    {'tokens': ['I', 'went', 'to', 'the', 'store', 'and', 'bought', 'some', '菜', '.'],
     'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'zh', 'en']},

    # Utterance 2: Mostly Chinese, switch to En phrase (intra-sentential)
    {'tokens': ['我', '想', '吃', 'some', 'sea', 'food', '今天', '.'],
     'langs': ['zh', 'zh', 'zh', 'en', 'en', 'en', 'zh', 'en']},

    # Utterance 3: Purely English (no switch)
    {'tokens': ['The', 'weather', 'is', 'nice', 'today', '.'],
     'langs': ['en', 'en', 'en', 'en', 'en', 'en']},

    # Utterance 4: Purely Chinese (no switch)
    {'tokens': ['你', '去', '哪里', '了吗', '?', '我', '在', '等你', '.'],
     'langs': ['zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh', 'zh']},

    # Utterance 5: Inter-sentential switch and back
    {'tokens': ['Hello', 'my', 'friend', '.', '我们', '开始', '吧', '.'],
     'langs': ['en', 'en', 'en', 'en', 'zh', 'zh', 'zh', 'zh']},
]

df = pd.DataFrame(mock_data)
print("1. Mock Dataset Created (Utterance View):")
print(df)

1. Mock Dataset Created (Utterance View):
                                              tokens  \
0  [I, went, to, the, store, and, bought, some, 菜...   
1                  [我, 想, 吃, some, sea, food, 今天, .]   
2                 [The, weather, is, nice, today, .]   
3                     [你, 去, 哪里, 了吗, ?, 我, 在, 等你, .]   
4               [Hello, my, friend, ., 我们, 开始, 吧, .]   

                                      langs  
0  [en, en, en, en, en, en, en, en, zh, en]  
1          [zh, zh, zh, en, en, en, zh, en]  
2                  [en, en, en, en, en, en]  
3      [zh, zh, zh, zh, zh, zh, zh, zh, zh]  
4          [en, en, en, en, zh, zh, zh, zh]  


2. Feature and Label Extraction (Token-Level)

This function extracts the token-level features (X) and the target label (Y).

In [3]:
# Define column names for clarity (even though they aren't used in the DF directly)
TOKEN_COL = 'tokens'
LANG_COL = 'langs'

def extract_features_and_labels(df, token_col_name, lang_col_name):
    """
    Iterates through tokens to create X (features) and Y (next token's language).
    """
    data = []

    for _, row in df.iterrows():
        tokens = row[token_col_name]
        langs = row[lang_col_name]

        # Iterate up to the second-to-last token, as we predict the *next* token's language
        for i in range(len(tokens) - 1):
            current_token = tokens[i]
            current_lang = langs[i]
            next_lang = langs[i+1] # This is our target (Y)

            # --- Feature Engineering (X): Contextual and Lexical Features ---
            feature_dict = {
                # 1. Current word (Lexical feature)
                'current_word': str(current_token).lower(),
                # 2. Current language (Crucial contextual feature)
                'current_lang': current_lang,
                # 3. Context: Previous word's language (L-1 language)
                'prev_lang': langs[i-1] if i > 0 else 'START',
                # 4. Lexical feature (Token length)
                'word_len': len(str(current_token)),
            }

            data.append({
                'features': feature_dict,
                'target_lang': next_lang
            })

    return pd.DataFrame(data)

print("\n2. Extracting token-level features...")
token_df = extract_features_and_labels(df, TOKEN_COL, LANG_COL)
print(f"Total token-level instances for prediction: {len(token_df)}")
print("\nSample of Extracted Token Data:")
print(token_df.head(5))


2. Extracting token-level features...
Total token-level instances for prediction: 36

Sample of Extracted Token Data:
                                            features target_lang
0  {'current_word': 'i', 'current_lang': 'en', 'p...          en
1  {'current_word': 'went', 'current_lang': 'en',...          en
2  {'current_word': 'to', 'current_lang': 'en', '...          en
3  {'current_word': 'the', 'current_lang': 'en', ...          en
4  {'current_word': 'store', 'current_lang': 'en'...          en


3. Feature Vectorization and Data Split

We vectorize the extracted features and split the data for training and testing.

In [4]:
print("\n3. Vectorizing features and splitting data...")

# Separate features (X_raw) and labels (Y)
X_raw = token_df['features']
Y = token_df['target_lang']

# Convert features into a numerical feature matrix (One-Hot Encoding of categorical features)
vectorizer = DictVectorizer(sparse=False)
X = vectorizer.fit_transform(X_raw)

# Get the unique language labels (Classes)
unique_langs = sorted(Y.unique().tolist())

print(f"Unique target languages (Classes): {unique_langs}")
print(f"Feature matrix shape: {X.shape}")

# Split the data (using stratify for imbalance)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42, stratify=Y
)

# Check class imbalance
lang_counts = pd.Series(Y_train).value_counts()
print(f"\nTraining set class distribution:\n{lang_counts}")


3. Vectorizing features and splitting data...
Unique target languages (Classes): ['en', 'zh']
Feature matrix shape: (36, 39)

Training set class distribution:
target_lang
en    14
zh    11
Name: count, dtype: int64


4. Model Training and Evaluation (Logistic Regression Baseline)

We train the model and generate the evaluation report, focusing on the F1-Score to measure success on the minority class.

In [5]:
print("\n4. Training Logistic Regression Baseline Model...")

# Initialize the Logistic Regression model (Classic ML Baseline)
# n_jobs=-1 uses all available processors, 'liblinear' is a good robust solver.
model = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000, n_jobs=-1)

# Train the model
model.fit(X_train, Y_train)

# Make predictions on the test set
Y_pred = model.predict(X_test)

# --- Evaluation ---
print("\n--- Model Evaluation (Classification Report) ---")
# The report provides Precision, Recall, and F1-score for each class
report = classification_report(Y_test, Y_pred, target_names=unique_langs, digits=4, zero_division=0)
print(report)

# Note to Students: In a real-world scenario, you would focus on the F1-score
# for the minority language class to truly assess code-switch prediction success.


4. Training Logistic Regression Baseline Model...

--- Model Evaluation (Classification Report) ---
              precision    recall  f1-score   support

          en     0.8333    0.8333    0.8333         6
          zh     0.8000    0.8000    0.8000         5

    accuracy                         0.8182        11
   macro avg     0.8167    0.8167    0.8167        11
weighted avg     0.8182    0.8182    0.8182        11



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(
